# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a [Croissant](https://mlcommons.org/croissant/) dataset using the `mlcroissant` library.

### Dataset Source
- Croissant Schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Dataset Title: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Explore available record sets, their fields, and `@id`s.

Below, we list all record sets and their available fields. We reference every entity (record sets, fields, columns) by their `@id`.

In [ ]:
# List all available record sets and their fields
record_sets = dataset.metadata.recordSet
if not record_sets or len(record_sets) == 0:
    print("No record sets found. Please check the dataset definition.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        # Ensure fields is always a list
        if isinstance(fields, dict):
            fields = [fields]
        elif isinstance(fields, list):
            pass
        else:
            fields = []
        print("  Fields:")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            print(f"    - Field @id: {field_id}")
        print()

## 3. Data Extraction
Load records from each record set into pandas DataFrames for analysis.

We will extract the first available record set (by its `@id`) as an example. You can add more as needed.

In [ ]:
# If at least one record set is found, extract all records for it as a DataFrame
dataframes = {}
if not record_sets or len(record_sets) == 0:
    print("No record sets to extract.")
else:
    # You can add more record set @id's here as needed
    record_set_ids = [rs['@id'] for rs in record_sets]
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"No records found for record set {record_set_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))
    # Pick one for demonstration
    example_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

## 4. Exploratory Data Analysis (EDA)
You can now process the data by filtering, normalizing, and grouping.

Below, we select a numeric field and group field by their `@id` from the selected record set, if available. **Replace the field IDs with those appropriate for your dataset as found in section 2.**

In [ ]:
import numpy as np

# Example: Find a numeric field and a category/group field from the columns
if dataframes:
    df = dataframes[example_record_set_id]
    # Try to auto-detect numeric columns
    numeric_field = None
    group_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field:
            group_field = col
            break

    if numeric_field is None:
        print("No numeric field found in the selected record set. Please update the field selection.")
    else:
        threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else None
        if threshold is None:
            print("Numeric field is not float/int. Please convert or select another field.")
        else:
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.3f} (mean):\n")
            display(filtered_df.head())

            # Normalize the numeric field
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
            print(f"Normalized {numeric_field} for filtered records (z-score):")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by group_field if available
            if group_field and group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by '{group_field}':")
                display(grouped_df.head())
            else:
                print("No suitable group-by field found.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships in the data.

**Tip:** You can adjust field and group names based on your actual data schema (`@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(y=df[numeric_field], x=df[group_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No data to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a Croissant (FAIR) dataset with `mlcroissant`
- List available record sets and fields by their `@id`
- Extract data into DataFrames, reference fields dynamically by their IDs
- Perform basic EDA (filtering, normalization, grouping)
- Visualize distributions and relationships in the dataset

**Next steps:** Adapt this notebook by selecting specific record sets, fields, and groupings most relevant to your research questions, referencing their canonical `@id`s as shown above for reproducibility.